# Universal Silver Transformation

One notebook for any dataset registered as active in `metadata/schema_config/`.

The notebook:
1. Reads the selected dataset's active schema from the shared schema-config Delta table.
2. Reads the corresponding Bronze Delta path.
3. Standardizes configured source columns to their registered schema types.
4. Applies optional dataset-specific quality rules.
5. Writes valid rows to `silver/{dataset_name}/`.
6. Writes invalid rows to `quarantine/silver/{dataset_name}/`.
7. Verifies row counts after writing.

**Write behavior:** This is a full refresh. Each successful run overwrites that dataset's Silver
and quarantine outputs. Bronze and the shared manifest are not modified.

**Before use:** Ensure the dataset has exactly one active row in schema_config and a Bronze Delta
table at the expected path. Add business rules to `DATASET_RULES` only after verifying them.

In [0]:
from datetime import datetime, timezone
import re
import uuid

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

In [0]:
# Storage paths: keep these aligned with Universal Bronze.
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"

# Select the dataset to process. Use the exact dataset_name registered in schema_config.
dbutils.widgets.text("dataset_name", "google_ads_campaigns", "Dataset name")
DATASET_NAME = dbutils.widgets.get("dataset_name").strip().lower()

if not DATASET_NAME:
    raise ValueError("dataset_name cannot be empty.")

if not re.fullmatch(r"[a-z][a-z0-9_]*", DATASET_NAME):
    raise ValueError(
        "dataset_name must contain lowercase letters, digits, and underscores."
    )

BRONZE_PATH = BASE_PATH + f"bronze/{DATASET_NAME}/"
SILVER_PATH = BASE_PATH + f"silver/{DATASET_NAME}/"
QUARANTINE_PATH = BASE_PATH + f"quarantine/silver/{DATASET_NAME}/"

SILVER_RUN_ID = str(uuid.uuid4())
SILVER_PROCESSED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

print(f"Dataset: {DATASET_NAME}")
print(f"Schema config: {SCHEMA_CONFIG_PATH}")
print(f"Bronze input: {BRONZE_PATH}")
print(f"Silver output: {SILVER_PATH}")
print(f"Quarantine output: {QUARANTINE_PATH}")
print(f"Silver run ID: {SILVER_RUN_ID}")

Dataset: google_ads_campaigns
Schema config: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/schema_config/
Bronze input: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/google_ads_campaigns/
Silver output: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/silver/google_ads_campaigns/
Quarantine output: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/quarantine/silver/google_ads_campaigns/
Silver run ID: d7e346d3-d6af-4b53-9d20-77cdf955a6c2


In [0]:
# Load exactly one active schema for the selected dataset.
if not DeltaTable.isDeltaTable(spark, SCHEMA_CONFIG_PATH):
    raise RuntimeError(
        f"Schema configuration Delta table not found: {SCHEMA_CONFIG_PATH}"
    )

schema_config_df = spark.read.format("delta").load(SCHEMA_CONFIG_PATH)

required_config_columns = {
    "dataset_name", "source_system", "schema_ddl", "schema_version", "is_active"
}
missing_config_columns = required_config_columns - set(schema_config_df.columns)
if missing_config_columns:
    raise RuntimeError(
        f"Schema config is missing required columns: {sorted(missing_config_columns)}"
    )

active_rows = (
    schema_config_df
    .filter(
        (F.lower(F.trim(F.col("dataset_name"))) == DATASET_NAME)
        & (F.col("is_active") == F.lit(True))
    )
    .select("dataset_name", "source_system", "schema_ddl", "schema_version")
    .collect()
)

if len(active_rows) != 1:
    raise ValueError(
        f"Expected exactly one active schema for '{DATASET_NAME}', "
        f"but found {len(active_rows)}. Register or correct the active schema first."
    )

SCHEMA_SOURCE_SYSTEM = active_rows[0]["source_system"]
SCHEMA_DDL = active_rows[0]["schema_ddl"]
SCHEMA_VERSION = active_rows[0]["schema_version"]

if not SCHEMA_DDL or not SCHEMA_DDL.strip():
    raise ValueError(f"schema_ddl is empty for '{DATASET_NAME}'.")

SOURCE_SCHEMA = StructType.fromDDL(SCHEMA_DDL)
SOURCE_FIELDS = {field.name: field.dataType for field in SOURCE_SCHEMA.fields}

if not SOURCE_FIELDS:
    raise ValueError(f"No source fields found in schema_ddl for '{DATASET_NAME}'.")

print(f"Schema version: {SCHEMA_VERSION}")
print(f"Schema source system: {SCHEMA_SOURCE_SYSTEM}")
print(f"Configured source columns: {len(SOURCE_FIELDS)}")
print(f"Schema DDL: {SCHEMA_DDL}")

Schema version: 1
Schema source system: google_ads
Configured source columns: 18
Schema DDL: customer_id STRING,
campaign_id STRING,
campaign_resource_name STRING,
campaign_name STRING,
campaign_status STRING,
advertising_channel_type STRING,
advertising_channel_sub_type STRING,
campaign_budget_resource_name STRING,
bidding_strategy_type STRING,
start_date STRING,
end_date STRING,
campaign_primary_status STRING,
serving_status STRING,
target_country STRING,
business_brand STRING,
product_category STRING,
business_objective STRING,
source_extract_timestamp STRING


In [0]:
# Optional dataset-specific quality rules.
#
# Only add rules that have been confirmed for the real dataset.
# Datasets absent from this dictionary still receive generic schema checks/casts.
#
# required_non_null: source columns that must be populated.
# Example below follows the current Google Ads campaigns test schema and should be
# retained only if customer_id and campaign_id are required by your actual source contract.
DATASET_RULES = {
    "google_ads_campaigns": {
        "required_non_null": ["customer_id", "campaign_id"],
    },
}

RULES = DATASET_RULES.get(DATASET_NAME, {})
REQUIRED_NON_NULL = RULES.get("required_non_null", [])

unknown_required_columns = [c for c in REQUIRED_NON_NULL if c not in SOURCE_FIELDS]
if unknown_required_columns:
    raise ValueError(
        f"Quality rules reference columns not present in schema_config: {unknown_required_columns}"
    )

In [0]:
# Read the selected Bronze Delta table.
if not DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    raise RuntimeError(
        f"Bronze Delta table not found at {BRONZE_PATH}. "
        "Run Universal Bronze successfully for this dataset first."
    )

bronze_df = spark.read.format("delta").load(BRONZE_PATH)
bronze_columns = set(bronze_df.columns)

missing_source_columns = [c for c in SOURCE_FIELDS if c not in bronze_columns]
if missing_source_columns:
    raise RuntimeError(
        f"Bronze is missing columns registered in schema_config: {missing_source_columns}. "
        f"Bronze columns: {bronze_df.columns}"
    )

print(f"Bronze row count: {bronze_df.count()}")
print("Bronze schema:")
bronze_df.printSchema()

In [0]:
# Apply generic standardization using schema_config data types.
# Keep all Bronze metadata columns unchanged. Only fields defined in schema_ddl are standardized.
# Temporary raw values let us identify non-empty values that fail a cast.
working_df = bronze_df
raw_backup_columns = {}
cast_failure_conditions = []

for field_name, spark_type in SOURCE_FIELDS.items():
    raw_backup = f"__silver_raw_{field_name}"
    raw_backup_columns[field_name] = raw_backup

    working_df = working_df.withColumn(
        raw_backup,
        F.col(field_name).cast("string")
    )

    if spark_type.simpleString() == "string":
        cleaned = F.when(
            F.length(F.trim(F.col(field_name))) == 0,
            F.lit(None).cast("string")
        ).otherwise(F.trim(F.col(field_name)))
        working_df = working_df.withColumn(field_name, cleaned)
    else:
        target_type_sql = spark_type.simpleString().upper()
        working_df = working_df.withColumn(
            field_name,
            F.expr(f"try_cast(`{field_name}` AS {target_type_sql})")
        )
        raw_column = raw_backup_columns[field_name]
        cast_failure_conditions.append(
            (
                F.col(raw_column).isNotNull()
                & (F.length(F.trim(F.col(raw_column))) > 0)
                & F.col(field_name).isNull(),
                f"CAST_FAILED:{field_name}_TO_{target_type_sql}",
            )
        )

# Build invalid-row reasons from configured required fields and failed casts.
invalid_conditions = []

for field_name in REQUIRED_NON_NULL:
    invalid_conditions.append(
        (
            F.col(field_name).isNull()
            | (F.length(F.trim(F.col(field_name).cast("string"))) == 0),
            f"REQUIRED_FIELD_MISSING:{field_name}",
        )
    )

invalid_conditions.extend(cast_failure_conditions)

reason_expressions = [
    F.when(condition, F.lit(reason))
    for condition, reason in invalid_conditions
]

working_df = working_df.withColumn(
    "silver_invalid_reason",
    F.concat_ws("; ", *reason_expressions) if reason_expressions else F.lit("")
)

invalid_df = working_df.filter(F.length(F.col("silver_invalid_reason")) > 0)
valid_df = working_df.filter(F.length(F.col("silver_invalid_reason")) == 0)

# Remove temporary columns. Retain invalid reason only in quarantine.
for raw_column in raw_backup_columns.values():
    valid_df = valid_df.drop(raw_column)
    invalid_df = invalid_df.drop(raw_column)

valid_df = (
    valid_df
    .drop("silver_invalid_reason")
    .withColumn("silver_processed_at", F.lit(SILVER_PROCESSED_AT).cast("timestamp"))
    .withColumn("silver_run_id", F.lit(SILVER_RUN_ID))
    .withColumn("silver_schema_version", F.lit(int(SCHEMA_VERSION)))
)

invalid_df = (
    invalid_df
    .withColumn("silver_processed_at", F.lit(SILVER_PROCESSED_AT).cast("timestamp"))
    .withColumn("silver_run_id", F.lit(SILVER_RUN_ID))
    .withColumn("silver_schema_version", F.lit(int(SCHEMA_VERSION)))
)

In [0]:
# Quality summary and row reconciliation.
source_count = bronze_df.count()
valid_count = valid_df.count()
invalid_count = invalid_df.count()

print("Silver quality summary")
print(f"Dataset: {DATASET_NAME}")
print(f"Bronze source rows: {source_count}")
print(f"Valid Silver rows: {valid_count}")
print(f"Quarantined rows: {invalid_count}")
print(f"Reconciliation: {valid_count} + {invalid_count} = {valid_count + invalid_count}")

if valid_count + invalid_count != source_count:
    raise RuntimeError(
        "Row reconciliation failed: valid + quarantined rows do not equal Bronze source rows."
    )

print("Sample valid rows:")
display(valid_df.limit(20))

if invalid_count > 0:
    print("Sample quarantined rows:")
    display(invalid_df.limit(20))
else:
    print("No invalid rows found under the configured quality rules.")

In [0]:
# Full refresh writes.
# mode('overwrite') creates the Delta path on the first run and replaces it on later runs.
# This is path-based Delta storage; a metastore CREATE TABLE is not required.
# Only this dataset's Silver/quarantine paths are written. Bronze and manifest remain untouched.
if not DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    raise RuntimeError("Bronze Delta table disappeared before write; aborting.")

(
    valid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_PATH)
)

print("Silver and quarantine writes completed.")
print(f"Silver path: {SILVER_PATH}")
print(f"Quarantine path: {QUARANTINE_PATH}")

In [0]:
# Verify the persisted counts.
silver_check_df = spark.read.format("delta").load(SILVER_PATH)
quarantine_check_df = spark.read.format("delta").load(QUARANTINE_PATH)

silver_written_count = silver_check_df.count()
quarantine_written_count = quarantine_check_df.count()

print(f"Verified Silver row count: {silver_written_count}")
print(f"Verified quarantine row count: {quarantine_written_count}")

if silver_written_count != valid_count:
    raise RuntimeError(
        f"Silver count mismatch: expected {valid_count}, found {silver_written_count}."
    )

if quarantine_written_count != invalid_count:
    raise RuntimeError(
        f"Quarantine count mismatch: expected {invalid_count}, found {quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != source_count:
    raise RuntimeError(
        "Post-write reconciliation failed: Silver + quarantine counts do not equal Bronze count."
    )

print(
    f"Universal Silver completed successfully for '{DATASET_NAME}' "
    f"(schema version {SCHEMA_VERSION})."
)